# Customer Dimension Data Processing Pipeline
## Bronze → Silver → Gold → Master Dimension

### 🚀 Ready to Run!

**1. S3 Access:**
   - ✅ Unity Catalog credentials configured!
   - Credential: `db_s3_credentials_databricks-s3-ingest-9db34`
   - No manual setup needed

**2. Data Source:**
   - S3 Bucket: `sportx-bar` (region: ap-southeast-2)
   - Path: `s3://sportx-bar/customers/*.csv`
   - All CSV files in the folder will be loaded and concatenated

**3. Output files:**
   All processed data will be saved to the `data/` subfolder:
   - `data/bronze_customers.csv` - Raw data with metadata
   - `data/silver_customers.csv` - Cleaned and transformed data
   - `data/gold_sb_dim_customers.csv` - Business dimension table
   - `data/gold_dim_customers.csv` - Master dimension table

**4. Run cells sequentially** from top to bottom.

---

In [0]:
import pandas as pd
import numpy as np
import os
from datetime import datetime

print("✓ Core libraries imported")

In [0]:
# Define parameters
catalog = "fmcg"
data_source = "customers"

# S3 bucket with Unity Catalog credentials
s3_bucket = "sportx-bar"  # Using Unity Catalog credential: db_s3_credentials_databricks-s3-ingest-9db34

print(f"Catalog: {catalog}")
print(f"Data source: {data_source}")
print(f"S3 path: s3://{s3_bucket}/{data_source}/*.csv")

# Create data directory for outputs
os.makedirs("data", exist_ok=True)


In [0]:
# Read CSV from S3 using Spark (automatically uses Unity Catalog credentials)
# Then convert to pandas for processing

# Build S3 path
s3_path = f's3://{s3_bucket}/{data_source}/*.csv'
print(f"Reading from S3: {s3_path}")
print(f"Using Unity Catalog credential: db_s3_credentials_databricks-s3-ingest-9db34")

try:
    # Read CSV from S3 using Spark (UC credentials work automatically)
    df_spark = spark.read.csv(s3_path, header=True, inferSchema=True)
    
    # Convert to pandas DataFrame
    df = df_spark.toPandas()
    
    print(f"\n✓ Successfully loaded {len(df)} rows")
    
    # Add Bronze metadata columns
    df["read_timestamp"] = pd.Timestamp.now()
    df["file_name"] = f"{data_source}.csv"
    df["file_size"] = None
    
    print(f"\nColumns: {list(df.columns)}")
    print(f"Shape: {df.shape}")
    
except Exception as e:
    print(f"\n✗ Error reading from S3: {e}")
    print(f"\nTroubleshooting:")
    print(f"  1. Verify file exists at: {s3_path}")
    print(f"  2. Check Unity Catalog external location is configured for: s3://{s3_bucket}/")
    print(f"  3. Ensure credential has permissions: s3:GetObject, s3:ListBucket")
    raise

In [0]:
# Display first 10 rows
print("\nBronze data preview:")
display(df.head(10))

In [0]:
# Save Bronze layer to CSV
output_path = f"data/bronze_{data_source}.csv"
df.to_csv(output_path, index=False)

print(f"✓ Bronze layer saved to: {output_path}")
print(f"  Rows: {len(df)}")
print(f"  Columns: {len(df.columns)}")

In [0]:
# Load Bronze data from CSV
df_bronze = pd.read_csv(f"data/bronze_{data_source}.csv")
print(f"Loaded Bronze data: {len(df_bronze)} rows")
print(f"\nFirst 10 rows:")
display(df_bronze.head(10))

In [0]:
# Check for duplicate customer_ids
df_duplicates = df_bronze.groupby('customer_id').size().reset_index(name='count')
df_duplicates = df_duplicates[df_duplicates['count'] > 1]

print(f"Duplicate customer_ids: {len(df_duplicates)}")
if len(df_duplicates) > 0:
    display(df_duplicates)

In [0]:
# Remove duplicate customer_ids (keep first occurrence)
print(f'Rows before duplicates dropped: {len(df_bronze)}')
df_silver = df_bronze.drop_duplicates(subset=['customer_id'], keep='first')
print(f'Rows after duplicates dropped: {len(df_silver)}')

In [0]:
# Check for customer names with leading/trailing spaces
df_with_spaces = df_silver[df_silver['customer_name'] != df_silver['customer_name'].str.strip()]
print(f"Rows with spaces: {len(df_with_spaces)}")
if len(df_with_spaces) > 0:
    display(df_with_spaces)

In [0]:
# Trim leading/trailing spaces from customer_name
df_silver['customer_name'] = df_silver['customer_name'].str.strip()

In [0]:
# Verify trim was successful
df_with_spaces = df_silver[df_silver['customer_name'] != df_silver['customer_name'].str.strip()]
print(f"Rows with spaces after trim: {len(df_with_spaces)}")

In [0]:
# Show all distinct city values
print("Distinct cities (including typos):")
print(df_silver['city'].unique())

In [0]:
# Fix city typos and standardize
city_mapping = {
    'Bengaluruu': 'Bengaluru',
    'Bengalore': 'Bengaluru',
    'Hyderabadd': 'Hyderabad',
    'Hyderbad': 'Hyderabad',
    'NewDelhi': 'New Delhi',
    'NewDheli': 'New Delhi',
    'NewDelhee': 'New Delhi'
}

# Apply mapping
df_silver['city'] = df_silver['city'].replace(city_mapping)

# Only keep allowed cities, set others to None
allowed = ["Bengaluru", "Hyderabad", "New Delhi"]
df_silver.loc[~df_silver['city'].isin(allowed), 'city'] = None

print("\nDistinct cities after correction:")
print(df_silver['city'].unique())

In [0]:
# Apply title case to customer names
df_silver['customer_name'] = df_silver['customer_name'].str.title()

print("\nDistinct customer names (title case):")
print(df_silver['customer_name'].unique())

In [0]:
# Show customers with null cities
df_null_cities = df_silver[df_silver['city'].isna()]
print(f"\nRows with null city: {len(df_null_cities)}")
if len(df_null_cities) > 0:
    display(df_null_cities)

In [0]:
# Identify specific customers with missing cities
null_customer_names = ['Sprintx Nutrition', 'Zenathlete Foods', 'Primefuel Nutrition', 'Recovery Lane']
df_to_fix = df_silver[df_silver['customer_name'].isin(null_customer_names)]

print(f"\nCustomers needing city fix: {len(df_to_fix)}")
if len(df_to_fix) > 0:
    display(df_to_fix)

In [0]:
# Manual city fixes for specific customers
customer_city_fix = {
    789403: "New Delhi",      # Sprintx Nutrition
    789420: "Bengaluru",      # Zenathlete Foods
    789521: "Hyderabad",      # Primefuel Nutrition
    789603: "Hyderabad"       # Recovery Lane
}

print("\nCity fix mapping:")
for customer_id, city in customer_city_fix.items():
    print(f"  {customer_id}: {city}")

In [0]:
# Apply city fixes for specific customers
for customer_id, city in customer_city_fix.items():
    df_silver.loc[df_silver['customer_id'] == customer_id, 'city'] = city

print("\n✓ Applied city fixes")

# Verify no more null cities for those customers
df_fixed = df_silver[df_silver['customer_id'].isin(customer_city_fix.keys())]
print(f"\nFixed customers:")
display(df_fixed[['customer_id', 'customer_name', 'city']])

In [0]:
# Cast customer_id to string for consistency
df_silver['customer_id'] = df_silver['customer_id'].astype(str)

print("\nData types:")
print(df_silver.dtypes)

In [0]:
# Create customer column: "CustomerName-City" or "CustomerName-Unknown"
df_silver['customer'] = df_silver.apply(
    lambda row: f"{row['customer_name']}-{row['city'] if pd.notna(row['city']) else 'Unknown'}",
    axis=1
)

# Add static attributes
df_silver['market'] = 'India'
df_silver['platform'] = 'Sports Bar'
df_silver['channel'] = 'Acquisition'

print("\n✓ Created customer column and added static fields")
print(f"\nColumns: {list(df_silver.columns)}")

In [0]:
# Display final Silver data
print("\nSilver data preview:")
display(df_silver.head(10))

In [0]:
# Save Silver layer to CSV
output_path = f"data/silver_{data_source}.csv"
df_silver.to_csv(output_path, index=False)

print(f"\n✓ Silver layer saved to: {output_path}")
print(f"  Rows: {len(df_silver)}")
print(f"  Columns: {len(df_silver.columns)}")

In [0]:
# Gold layer: Load Silver data and select required columns
df_silver_loaded = pd.read_csv(f"data/silver_{data_source}.csv")
print(f"Loaded Silver data: {len(df_silver_loaded)} rows")

# Select required columns for Gold layer
required_cols = ["customer_id", "customer_name", "city", "customer", "market", "platform", "channel"]
df_gold = df_silver_loaded[required_cols].copy()

print(f"\nGold layer preview:")
display(df_gold.head(5))

In [0]:
# Save Gold layer to CSV
output_path = f"data/gold_sb_dim_{data_source}.csv"
df_gold.to_csv(output_path, index=False)

print(f"\n✓ Gold layer saved to: {output_path}")
print(f"  Rows: {len(df_gold)}")
print(f"  Columns: {len(df_gold.columns)}")

In [0]:
# Master Dimension: Load child customers from Gold layer
df_child_customers = pd.read_csv(f"data/gold_sb_dim_{data_source}.csv")

# Rename customer_id to customer_code for master dimension
df_child_customers = df_child_customers.rename(columns={'customer_id': 'customer_code'})

# Select required columns
df_child_customers = df_child_customers[['customer_code', 'customer', 'market', 'platform', 'channel']].copy()

print(f"\nChild customers loaded: {len(df_child_customers)} rows")
display(df_child_customers.head(5))

In [0]:
# Merge with existing master dimension table (upsert logic)
master_path = f"data/gold_dim_{data_source}.csv"

try:
    # Load existing master table
    df_master = pd.read_csv(master_path)
    print(f"\nExisting master table found: {len(df_master)} rows")
    
    # Remove rows from master that exist in child (will be replaced)
    customer_codes_to_update = df_child_customers['customer_code'].unique()
    df_master = df_master[~df_master['customer_code'].isin(customer_codes_to_update)]
    print(f"Rows after removing updated customers: {len(df_master)}")
    
    # Append child customers
    df_merged = pd.concat([df_master, df_child_customers], ignore_index=True)
    print(f"\nRows after merge: {len(df_merged)}")
    
except FileNotFoundError:
    print("\nNo existing master table found. Creating new one.")
    df_merged = df_child_customers.copy()

# Save updated master dimension
df_merged.to_csv(master_path, index=False)

print(f"\n✓ Master dimension table updated: {master_path}")
print(f"  Total rows: {len(df_merged)}")
print(f"  Child rows processed: {len(df_child_customers)}")

print("\n" + "="*60)
print("PROCESSING COMPLETE")
print("="*60)
print(f"Bronze: data/bronze_{data_source}.csv")
print(f"Silver: data/silver_{data_source}.csv")
print(f"Gold:   data/gold_sb_dim_{data_source}.csv")
print(f"Master: data/gold_dim_{data_source}.csv")